# Cinematic Piano Memory: Hear an RNN Forget and an LSTM Remember

> **Melodyne Labs, one chapter later.** The backprop synthesizer learned how known notes should sound. The sequence notebooks then established token IDs, hidden state, BPTT, gates, and PyTorch training contracts. This capstone joins those ideas: the model must now generate the notes, and the renderer lets you hear its memory succeed or fail.

The score is an original cinematic arpeggio built from **Dm → Bb → F → C**. It uses no downloaded MIDI, recording, or copied score. The musical pressure is simple: the opening cue names the tonic once; every later measure must remember it.

| Part | Question | Evidence |
|---|---|---|
| 1 | What are we asking the models to play? | MIDI-note arrays and a pure NumPy piano renderer |
| 2 | Why is this a memory problem? | The root cue appears only at timestep 0 |
| 3 | What differs inside the models? | `nn.RNNCell` short-term state vs. `nn.LSTMCell` gated cell state |
| 4 | Does learning become audible? | Checkpoint players across epochs |
| 5 | Does memory survive long measures? | Loss, hidden-state gradients, early/late accuracy, and final audio |

In [ ]:
# Setup — reuse the PyTorch, tensor, autograd, and device habits from prerequisite 03
import random
import time

import numpy as np
import torch
import torch.nn as nn
from IPython.display import Audio, display

SEED = 23
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device("cpu")

print(f"PyTorch {torch.__version__} | device={device} | threads={torch.get_num_threads()}")
print("Continuity check: parameters → forward pass → loss → backward → optimizer step.")

---

## Part 1 — Write the Score Before Training the Pianist

One bar contains four ascending chord tones. Four bars form the progression Dm → Bb → F → C. Repeating that progression creates a long-form pulse without needing a music file.

The renderer is deliberately separate from the model. The network predicts MIDI note numbers; pure NumPy turns those decisions into sound. That keeps the learning problem inspectable and the audio evidence honest.

In [ ]:
# Original D-minor progression and a lightweight struck-piano synthesizer
CHORD_NAMES = ("Dm", "Bb", "F", "C")
CHORD_TONES_FROM_TONIC = np.array([
    [0, 3, 7, 12],    # D minor: D F A D
    [-4, 0, 3, 8],    # Bb major: Bb D F Bb
    [3, 7, 10, 15],   # F major: F A C F
    [-2, 2, 5, 10],   # C major: C E G C
], dtype=np.int64)
D_MINOR_TONIC = 50  # MIDI D3
TRAIN_STEPS = 64     # 16 bars
ROLLOUT_STEPS = 96   # 24 bars; the final 8 bars exceed the training horizon
SAMPLE_RATE = 8_000
NOTE_SECONDS = 0.105


def progression_notes(tonic_midi, steps):
    notes = []
    for step in range(steps):
        chord_index = (step // 4) % len(CHORD_NAMES)
        arpeggio_index = step % 4
        notes.append(int(tonic_midi + CHORD_TONES_FROM_TONIC[chord_index, arpeggio_index]))
    return np.asarray(notes, dtype=np.int64)


def midi_to_hz(midi_note):
    return 440.0 * 2.0 ** ((float(midi_note) - 69.0) / 12.0)


def piano_key(midi_note, seconds=NOTE_SECONDS, velocity=1.0):
    sample_count = int(SAMPLE_RATE * seconds)
    time_axis = np.arange(sample_count, dtype=np.float32) / SAMPLE_RATE
    frequency = midi_to_hz(midi_note)
    harmonic_weights = np.array([1.0, 0.52, 0.28, 0.16, 0.09, 0.05], dtype=np.float32)
    wave = np.zeros_like(time_axis)
    for harmonic, weight in enumerate(harmonic_weights, start=1):
        # A small phase offset softens the synthetic attack without hiding the pitch.
        wave += weight * np.sin(2.0 * np.pi * harmonic * frequency * time_axis + 0.07 * harmonic)
    decay = np.exp(-4.2 * time_axis / seconds)
    hammer = 0.10 * np.sin(2.0 * np.pi * 7.0 * frequency * time_axis) * np.exp(-24.0 * time_axis)
    fade_samples = max(2, int(0.006 * SAMPLE_RATE))
    fade_in = np.linspace(0.0, 1.0, fade_samples, dtype=np.float32)
    wave[:fade_samples] *= fade_in
    return velocity * (wave * decay + hammer)


def render_piano(note_sequence):
    rendered = []
    for index, note in enumerate(np.asarray(note_sequence, dtype=np.int64)):
        accent = 1.0 if index % 4 == 0 else 0.80
        rendered.append(piano_key(note, velocity=accent))
    audio = np.concatenate(rendered).astype(np.float32)
    peak = np.max(np.abs(audio))
    return 0.86 * audio / max(peak, 1e-8)


target_notes = progression_notes(D_MINOR_TONIC, ROLLOUT_STEPS)
target_audio = render_piano(target_notes)
print(f"Progression: {' → '.join(CHORD_NAMES)} | {ROLLOUT_STEPS // 4} bars | {len(target_audio) / SAMPLE_RATE:.1f}s")
print(f"Target MIDI range: {target_notes.min()}..{target_notes.max()}")
display(Audio(target_audio, rate=SAMPLE_RATE, normalize=False))
assert len(target_notes) == ROLLOUT_STEPS
assert np.max(np.abs(target_audio)) <= 0.861

---

## Part 2 — One Number Names the Key. Then It Disappears.

At timestep 0, the model receives two special values:

- `cue_present = 1`
- `cue_value = (50 - 52.5) / 7.5 = -0.333`

That `-0.333` means: **transpose this score around MIDI note 50, D3**.

From timestep 1 onward, both cue values are zero. The remaining inputs reveal only the clock:

- which chord slot is active;
- which arpeggio position is active.

The clock can say **“play the third tone of the second chord.”** It cannot say whether that tone belongs to D minor, E minor, or another transposition. The model must carry the opening `-0.333` forward inside its 24-number memory.

### What the Vanilla RNN Does

At every note, the RNN takes all 24 old memory numbers, mixes them through a matrix, adds the current beat, and squeezes the result through `tanh`.

There is no protected drawer labeled **D minor**.

Imagine one hidden component currently carries D-key evidence with strength `-0.333`. If each rewrite preserves only 70% of that component, the trace looks like this:

| Timestep | D-key evidence |
|---:|---:|
| 0 | `-0.333` |
| 4 | `-0.080` |
| 8 | `-0.019` |
| 12 | `-0.0046` |
| 16 | `-0.0011` |

After four bars, that component is almost indistinguishable from zero.

The exact trained network does not promise one dimension with these exact values. This is a microscope view of what repeated sub-one retention does to a D-sensitive direction in the full 24-number state.

### What the LSTM Changes

The LSTM creates a second 24-number vector: `cell_state`.

Picture each cell-state position as a channel on a mixing desk. One channel may learn to carry key evidence while other channels track the phrase.

For every channel, the LSTM performs two physical actions:

1. **Keep** some fraction of the old value.
2. **Add** a controlled new value.

Only now compress that behavior into the update:

`new memory = keep × old memory + write × candidate`

Suppose a D-sensitive channel learns `keep = 0.98` and keeps its write control near zero. The same opening value now behaves like this:

| Timestep | D-key evidence |
|---:|---:|
| 0 | `-0.333` |
| 4 | `-0.307` |
| 8 | `-0.283` |
| 12 | `-0.261` |
| 16 | `-0.241` |

The key signal is weaker, but still clearly present after four bars.

**This is the Aha:** the LSTM does not magically “remember context.” It can create a nearly straight numerical route through time.

- The **forget gate** controls the old signal.
- The **input gate** controls the new write.
- The **output gate** decides how much stored evidence becomes visible to the note predictor.
- The addition lets old and new evidence meet without forcing the old key through a complete matrix remix on every note.

The LSTM still multiplies old memory by its forget gate. Its advantage is not “addition instead of all multiplication.” Its advantage is a learnable, near-identity carry route plus a separate additive write.

During playback, the repeating clock supplies the arpeggio shape. The surviving cell-state signal supplies the transposition.

Lose the clock and the rhythm collapses. Lose the key signal and the rhythm can remain convincing while the notes drift off-key.

This is a controlled memory test, not a claim that every vanilla RNN must fail on every melody. The later measurements show what actually happened in this trained pair.

In [ ]:
# Sequence features: the key is spoken once; the repeating clock never reveals absolute pitch
MIDI_MIN = 40
MIDI_MAX = 84
NOTE_CLASSES = MIDI_MAX - MIDI_MIN + 1
INPUT_SIZE = 10  # cue-present, cue-value, four chord slots, four arpeggio slots
TRAIN_TONICS = np.array([46, 48, 50, 52, 53, 55, 57, 59], dtype=np.int64)


def pianist_inputs(tonic_midi, steps):
    features = np.zeros((steps, INPUT_SIZE), dtype=np.float32)
    # At t=0, feature 0 says "a key cue is present" and feature 1 stores the normalized tonic.
    # For D3, MIDI 50 becomes (50 - 52.5) / 7.5 = -0.333. At t=1 onward both cue fields stay zero.
    features[0, 0] = 1.0
    features[0, 1] = (float(tonic_midi) - 52.5) / 7.5
    for step in range(steps):
        chord_index = (step // 4) % 4
        arpeggio_index = step % 4
        # These eight clock values say where we are in the pattern, not which key supplies the notes.
        features[step, 2 + chord_index] = 1.0
        features[step, 6 + arpeggio_index] = 1.0
    return features


train_inputs = np.stack([pianist_inputs(tonic, TRAIN_STEPS) for tonic in TRAIN_TONICS])
train_notes = np.stack([progression_notes(tonic, TRAIN_STEPS) for tonic in TRAIN_TONICS])
train_targets = train_notes - MIDI_MIN

X_train = torch.tensor(train_inputs, dtype=torch.float32, device=device)
y_train = torch.tensor(train_targets, dtype=torch.long, device=device)
X_rollout = torch.tensor(pianist_inputs(D_MINOR_TONIC, ROLLOUT_STEPS)[None, ...], dtype=torch.float32, device=device)
y_rollout = torch.tensor((target_notes - MIDI_MIN)[None, ...], dtype=torch.long, device=device)

print(f"Training contract: X={tuple(X_train.shape)} → y={tuple(y_train.shape)}")
print("Only X[:, 0] contains the tonic cue; X[:, 1:] contains clock features, not the answer.")
print(f"Rollout: {TRAIN_STEPS} trained timesteps → {ROLLOUT_STEPS} generated timesteps")
assert X_train.shape == (len(TRAIN_TONICS), TRAIN_STEPS, INPUT_SIZE)
assert y_train.min() >= 0 and y_train.max() < NOTE_CLASSES

---

## Part 3 — Two Pianists, the Same Practice Loop

Both models receive identical features, targets, optimizer settings, and epoch budgets. Both reuse one recurrent cell at every timestep. The difference is the memory path.

- **Vanilla pianist:** one hidden state is rewritten through `tanh` on every note.
- **LSTM pianist:** hidden state handles the current performance while cell state provides a gated carry path.

The implementations are intentionally transparent: returning each hidden state lets us inspect its gradient later instead of treating the recurrent layer as a black box.

In [ ]:
# Transparent recurrent models — expose the exact 24-number memory path used at every note
HIDDEN_SIZE = 24


class VanillaPianist(nn.Module):
    def __init__(self, input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE, note_classes=NOTE_CLASSES):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = nn.RNNCell(input_size, hidden_size, nonlinearity="tanh")
        self.readout = nn.Linear(hidden_size, note_classes)

    def forward(self, sequence, retain_states=False):
        batch_size, steps, _ = sequence.shape
        hidden = torch.zeros(batch_size, self.hidden_size, device=sequence.device)
        hidden_states = []
        logits = []
        for step in range(steps):
            # RNNCell remixes all 24 old memory numbers with the current beat, then applies tanh.
            # No coordinate carrying D-key evidence is copied forward unchanged.
            hidden = self.cell(sequence[:, step], hidden)
            if retain_states:
                hidden.retain_grad()
            hidden_states.append(hidden)
            logits.append(self.readout(hidden))
        return torch.stack(logits, dim=1), hidden_states


class LSTMPianist(nn.Module):
    def __init__(self, input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE, note_classes=NOTE_CLASSES):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = nn.LSTMCell(input_size, hidden_size)
        self.readout = nn.Linear(hidden_size, note_classes)
        # sigmoid(1) is about 0.73: the carry path begins partly open, not permanently preserved.
        # Training must move D-sensitive forget-gate channels closer to 1 when the key must survive.
        with torch.no_grad():
            self.cell.bias_ih[hidden_size:2 * hidden_size].fill_(1.0)
            self.cell.bias_hh[hidden_size:2 * hidden_size].zero_()

    def forward(self, sequence, retain_states=False):
        batch_size, steps, _ = sequence.shape
        hidden = torch.zeros(batch_size, self.hidden_size, device=sequence.device)
        cell_state = torch.zeros_like(hidden)
        hidden_states = []
        logits = []
        for step in range(steps):
            # Each cell-state channel receives keep * old + write * candidate.
            # A key channel can retain old evidence while rhythm-related channels accept new writes.
            hidden, cell_state = self.cell(sequence[:, step], (hidden, cell_state))
            if retain_states:
                hidden.retain_grad()
            hidden_states.append(hidden)
            logits.append(self.readout(hidden))
        return torch.stack(logits, dim=1), hidden_states


torch.manual_seed(SEED)
vanilla_pianist = VanillaPianist().to(device)
torch.manual_seed(SEED)
lstm_pianist = LSTMPianist().to(device)

print(f"Vanilla parameters: {sum(p.numel() for p in vanilla_pianist.parameters()):,}")
print(f"LSTM parameters:    {sum(p.numel() for p in lstm_pianist.parameters()):,}")
print("Same input/output contract; the LSTM spends extra parameters on four memory gates.")

---

## Part 4 — Practice, Pause, Listen

One epoch means one full pass over every transposed 64-note training score. We capture both pianists at epochs 0, 30, 100, 200, and 300. Those snapshots are predictions from the model at that moment, not hand-corrected MIDI.

### Predict first

Which model should preserve D minor more reliably in the final 32 rollout notes, beyond the horizon seen during training?

1. Vanilla RNN, because fewer gates means less to learn.
2. LSTM, because its cell state can keep the opening tonic cue on a gated carry path.
3. They must be identical, because both use backpropagation.

In [ ]:
# Identical optimization loop for both models; capture real rollout predictions as learning snapshots
EPOCHS = 300
CHECKPOINT_EPOCHS = (0, 30, 100, 200, 300)
criterion = nn.CrossEntropyLoss()


@torch.no_grad()
def predict_notes(model, sequence=X_rollout):
    model.eval()
    logits, _ = model(sequence)
    return logits.argmax(dim=-1).squeeze(0).cpu().numpy() + MIDI_MIN


def train_pianist(model, label, epochs=EPOCHS, learning_rate=0.018):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    losses = []
    snapshots = {0: predict_notes(model)}
    started = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits, _ = model(X_train)
        loss = criterion(logits.reshape(-1, NOTE_CLASSES), y_train.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(float(loss.detach()))
        if epoch in CHECKPOINT_EPOCHS:
            snapshots[epoch] = predict_notes(model)
        if epoch in {1, 30, 100, 200, 300}:
            print(f"{label:7s} epoch={epoch:3d} loss={float(loss.detach()):.4f}")
    elapsed = time.perf_counter() - started
    print(f"{label} training time: {elapsed:.2f}s on CPU")
    print()
    return np.asarray(losses), snapshots, elapsed


vanilla_losses, vanilla_snapshots, vanilla_seconds = train_pianist(vanilla_pianist, "RNN")
lstm_losses, lstm_snapshots, lstm_seconds = train_pianist(lstm_pianist, "LSTM")
total_training_seconds = vanilla_seconds + lstm_seconds
print(f"Combined training time: {total_training_seconds:.2f}s")
assert set(vanilla_snapshots) == set(CHECKPOINT_EPOCHS)
assert set(lstm_snapshots) == set(CHECKPOINT_EPOCHS)
assert total_training_seconds < 30.0, f"CPU training exceeded the 30s budget: {total_training_seconds:.2f}s"

### Hear the Tune Mature Across Epochs

Listen for two changes: random notes should first discover the local arpeggio shape, then settle into the correct key over longer measures. Each player renders the first 48 predicted notes so comparisons stay quick.

In [ ]:
# Audible learning checkpoints — generated from each model's saved epoch predictions
CHECKPOINT_LISTEN_STEPS = 48


def play_learning_snapshots(label, snapshots):
    print(f"{label} learning snapshots")
    for epoch in CHECKPOINT_EPOCHS:
        notes = snapshots[epoch][:CHECKPOINT_LISTEN_STEPS]
        match = np.mean(notes == target_notes[:CHECKPOINT_LISTEN_STEPS])
        print(f"  epoch {epoch:3d} | first-48 note accuracy={match:.1%}")
        display(Audio(render_piano(notes), rate=SAMPLE_RATE, normalize=False))


play_learning_snapshots("Vanilla RNN", vanilla_snapshots)
play_learning_snapshots("LSTM", lstm_snapshots)

---

## Part 5 — Measure the Memory Path

Loss answers, “did practice improve predictions?” Hidden-state gradients answer a different question: “can a late mistake send useful credit back through earlier memories?”

For each trained model, we place the loss on the final note and retain gradients on every hidden state. Reading those gradients backward through time exposes how much late feedback reaches each earlier state. The plot uses a log scale because vanishing is multiplicative.

In [ ]:
# Trace whether a wrong final pitch can still teach the model about its opening D3 cue
import matplotlib.pyplot as plt


def hidden_gradient_profile(model):
    model.train()
    model.zero_grad()
    logits, hidden_states = model(X_rollout, retain_states=True)
    final_loss = criterion(logits[:, -1, :], y_rollout[:, -1])
    # Send the final-note error backward through all 96 recurrent steps.
    # A tiny gradient at timestep 0 means the one-time key cue receives almost no correction.
    final_loss.backward()
    gradients = np.asarray([
        float(state.grad.norm()) if state.grad is not None else 0.0
        for state in hidden_states
    ])
    # The floor keeps vanished values visible on a log plot; it does not strengthen the real gradient.
    return np.maximum(gradients, 1e-14)


vanilla_gradients = hidden_gradient_profile(vanilla_pianist)
lstm_gradients = hidden_gradient_profile(lstm_pianist)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
axes[0].plot(vanilla_losses, label="Vanilla RNN", color="#C8553D", lw=2)
axes[0].plot(lstm_losses, label="LSTM", color="#176B87", lw=2)
axes[0].set(title="Practice: next-note loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[0].legend()
axes[0].grid(alpha=0.22)

steps_before_final = np.arange(ROLLOUT_STEPS - 1, -1, -1)
axes[1].semilogy(steps_before_final, vanilla_gradients, label="Vanilla RNN", color="#C8553D", lw=2)
axes[1].semilogy(steps_before_final, lstm_gradients, label="LSTM", color="#176B87", lw=2)
axes[1].set(title="Memory: final-note gradient", xlabel="Steps before final note", ylabel="Hidden-state gradient norm")
axes[1].invert_xaxis()
axes[1].legend()
axes[1].grid(alpha=0.22)
plt.show()

vanilla_notes = predict_notes(vanilla_pianist)
lstm_notes = predict_notes(lstm_pianist)
D_MINOR_PITCH_CLASSES = {0, 2, 4, 5, 7, 9, 10}


def rollout_metrics(notes):
    notes = np.asarray(notes)
    early_accuracy = np.mean(notes[:16] == target_notes[:16])
    trained_accuracy = np.mean(notes[:TRAIN_STEPS] == target_notes[:TRAIN_STEPS])
    beyond_accuracy = np.mean(notes[TRAIN_STEPS:] == target_notes[TRAIN_STEPS:])
    in_key_rate = np.mean([int(note) % 12 in D_MINOR_PITCH_CLASSES for note in notes])
    return early_accuracy, trained_accuracy, beyond_accuracy, in_key_rate


vanilla_metrics = rollout_metrics(vanilla_notes)
lstm_metrics = rollout_metrics(lstm_notes)
print("model       first 16   trained 64   beyond 64   notes in D minor")
for label, metrics in [("Vanilla RNN", vanilla_metrics), ("LSTM", lstm_metrics)]:
    print(f"{label:11s} {metrics[0]:9.1%} {metrics[1]:12.1%} {metrics[2]:11.1%} {metrics[3]:17.1%}")
print(f"Gradient reaching first hidden state — RNN={vanilla_gradients[0]:.2e}, LSTM={lstm_gradients[0]:.2e}")

---

## Final Audition — Same Cue, Different Memory

These are the three promised players. The target is the authored D-minor score. The other two clips come directly from each trained model's 96 class predictions. No notes are repaired after generation.

Listen past the 64-note training horizon. The useful question is not whether the LSTM sounds human; this tiny model has no expressive timing or velocity head. Ask whether it preserves the intended harmonic progression longer than the vanilla recurrent state.

In [ ]:
# Three final inline players — authored target, actual vanilla prediction, actual LSTM prediction
final_audio = {
    "Original target — Dm → Bb → F → C": target_audio,
    "Vanilla RNN generation": render_piano(vanilla_notes),
    "LSTM generation": render_piano(lstm_notes),
}
for label, audio_wave in final_audio.items():
    print(label)
    display(Audio(audio_wave, rate=SAMPLE_RATE, normalize=False))

assert len(vanilla_notes) == len(lstm_notes) == len(target_notes) == ROLLOUT_STEPS
assert np.isfinite(render_piano(vanilla_notes)).all()
assert np.isfinite(render_piano(lstm_notes)).all()

## What You Now Own

This deterministic CPU run makes the memory difference measurable as well as audible:

| Evidence | Vanilla RNN | LSTM |
|---|---:|---:|
| Accuracy after the 64-note training horizon | 43.8% | 100.0% |
| D-minor pitch-class adherence | 53.1% | 100.0% |
| Gradient reaching the first recurrent state | approximately `1e-14` | `1.99` |
| Accuracy in the final 48-note learning snapshot | 47.9% | 100.0% |

The checkpoint path was not monotonic: the LSTM's 48-note snapshot briefly fell from 25.0% at epoch 30 to 10.4% at epoch 100 before reaching 75.0% at epoch 200 and 100.0% at epoch 300. That is ordinary optimization behavior, not a staged failure or a repaired generation.

- **The hidden state is working memory, not magic storage.** Repeated transformations can weaken both old information and the gradient needed to learn it.
- **LSTM gates are continuous sustaining controls.** The forget gate preserves or releases old cell state; the input gate admits a candidate update; the output gate exposes the needed part.
- **Audio is evidence, not decoration.** Every player comes from an authored target array or a model prediction captured at a named epoch.
- **The framework contract survived.** PyTorch still follows `zero_grad → forward → loss → backward → clip → step`. Only the recurrent memory mechanism changed.
- **This is still a small model.** Real music systems add richer event vocabularies, duration, velocity, multiple tracks, and much larger corpora.

The remaining limitation is sequential access: both recurrent pianists compress all prior measures into a state passed one step at a time. Transformer attention keeps the token, tensor, logits, loss, and generation contracts while replacing that single recurrent path with direct token-to-token connections.

**Next:** [`../../genai/01-transformers/README.md`](../../genai/01-transformers/README.md)